# Dt 谱 × meta-gradient 金标准
**回答的问题：Dt 作为"下游贡献度代理"有没有价值。**

meta-grad（一阶）：∂L_task/∂w(t) ∝ −⟨∇L_task, ∇L_diff(t)⟩，是该 t 对下游任务真实增益的直接度量。Dt 谱与 meta-grad 谱的 Spearman/Pearson 相关性越高，Dt 越可信。

支持对多个 checkpoint 批量测量并汇总相关性演化。

In [1]:
import os, sys
import torch

# ---- 先在 notebook 所在目录（visual/）下把两个路径都算成绝对路径 ----
VISUAL = os.path.abspath('.')                                  # 内核启动时 cwd 就是 visual/
MMSEG  = os.path.abspath('../tasks/Segmentation/mmsegmentation')

os.chdir(MMSEG)
for p in [MMSEG, os.path.join(MMSEG, 'eden'), VISUAL]:
    if p not in sys.path:
        sys.path.insert(0, p)

# eden 包在服务器上部署名为 mydiffae：本地别名兼容
import eden, eden.diffae_mixin
sys.modules.setdefault('mydiffae', eden)
sys.modules.setdefault('mydiffae.diffae_mixin', eden.diffae_mixin)

# 注册 adapter（FCNWithDiffAEPlusPCGrad）；已注册过则跳过
try:
    import mmseg.models.segmentors.eden_adapter_fcn  # noqa
except KeyError:
    pass

import probe_core as pc

In [2]:
# ---- 配置：改成你的 config 与 ckpt 列表 ----
# fcn_diffae_unet 是 UNet 仪器的旧 run：CFG 用它 work_dir 里 train 时 dump 的
# 完整 config（mmseg 会把合并后的全量配置 dump 到 work_dir，口径与训练严格一致，
# 含当时用的 unet 超参与预测目标），再补 denoiser_type='unet'——旧 config 无此键，
# 而当前代码默认已切到 DiT；不补会把 UNet 权重错装进 DiT 壳（键全对不上、Dt 谱全 nan）。
import glob
RUN_DIR = 'work_dirs/eden/fcn_diffae_unet'
CFG = sorted(glob.glob(os.path.join(RUN_DIR, '*.py')))[0]   # run 目录里 dump 的 config
UNET_FIX = dict(denoiser_type='unet')
CKPTS = [
    f'{RUN_DIR}/iter_24000.pth',
    f'{RUN_DIR}/iter_40000.pth',
    f'{RUN_DIR}/iter_56000.pth',
    f'{RUN_DIR}/iter_72000.pth',
]
OUT = 'work_dirs/offline_probe/dt_meta'
NUM_IMAGES = 16
os.makedirs(OUT, exist_ok=True)

In [3]:
# 数据只取一次，所有 ckpt 共用（同图同分布，跨 ckpt 可比）
model, cfg = pc.build_model(CFG, cfg_overrides=UNET_FIX)   # 先建一次只为取数
batch = pc.build_val_batch(model, cfg, NUM_IMAGES)
del model; torch.cuda.empty_cache()

/data02/xujunkai/EDEN/tasks/Segmentation/mmsegmentation/mmseg/models/backbones/resnet.py:431: UserWarning: DeprecationWarning: pretrained is a deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is a deprecated, '
/data02/xujunkai/EDEN/tasks/Segmentation/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(
/data02/xujunkai/EDEN/tasks/Segmentation/mmsegmentation/mmseg/datasets/transforms/loading.py:84: UserWarning: `reduce_zero_label` will be deprecated, if you would like to ignore the zero label, please set `reduce_zero_label=True` when dataset initialized
  warnings.warn('`reduce_zero_label` will be deprecated, '


In [4]:
rows = []
for ckpt in CKPTS:
    model, _ = pc.build_model(CFG, ckpt, cfg_overrides=UNET_FIX)
    tag = os.path.splitext(os.path.basename(ckpt))[0]
    dt = pc.measure_dt_spectrum(model, batch)
    mg = pc.measure_meta_grad_spectrum(model, batch)
    fig, (rho_s, p_s, rho_p) = pc.plot_dt_vs_meta(
        dt, mg, title=tag,
        save_path=os.path.join(OUT, f"{tag}_dt_vs_meta.png"))
    rows.append((tag, rho_s, p_s, rho_p))
    print(f"{tag}: Spearman ρ={rho_s:.3f} (p={p_s:.3f}), Pearson r={rho_p:.3f}")
    del model; torch.cuda.empty_cache()
fig  # 显示最后一张

Loads checkpoint by local backend from path: work_dirs/eden/fcn_diffae_unet/iter_24000.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: diffae_ema_shadow, diffae_unet.time_embed.0.weight, diffae_unet.time_embed.0.bias, diffae_unet.time_embed.2.weight, diffae_unet.time_embed.2.bias, diffae_unet.input_conv.weight, diffae_unet.input_conv.bias, diffae_unet.down_blocks.0.norm1.weight, diffae_unet.down_blocks.0.norm1.bias, diffae_unet.down_blocks.0.conv1.weight, diffae_unet.down_blocks.0.conv1.bias, diffae_unet.down_blocks.0.time_emb_proj.1.weight, diffae_unet.down_blocks.0.time_emb_proj.1.bias, diffae_unet.down_blocks.0.cond_emb_proj.1.weight, diffae_unet.down_blocks.0.cond_emb_proj.1.bias, diffae_unet.down_blocks.0.norm2.weight, diffae_unet.down_blocks.0.norm2.bias, diffae_unet.down_blocks.0.conv2.weight, diffae_unet.down_blocks.0.conv2.bias, diffae_unet.down_blocks.1.op.weight, diffae_unet.down_blocks.1.op.bias, diffae_unet.down_blocks.2.norm1

/data02/xujunkai/EDEN/visual/probe_core.py:371: UserWarning: The label '_child1' of <matplotlib.lines.Line2D object at 0x7fdb6cbb0280> starts with '_'. It is thus excluded from the legend.
  ax.legend(lines, [l.get_label() for l in lines], fontsize=8)
/data02/xujunkai/EDEN/visual/probe_core.py:371: UserWarning: The label '_child1' of <matplotlib.lines.Line2D object at 0x7fdb6ca95eb0> starts with '_'. It is thus excluded from the legend.
  ax.legend(lines, [l.get_label() for l in lines], fontsize=8)


iter_24000: Spearman ρ=nan (p=nan), Pearson r=nan


/data02/xujunkai/EDEN/tasks/Segmentation/mmsegmentation/mmseg/models/backbones/resnet.py:431: UserWarning: DeprecationWarning: pretrained is a deprecated, please use "init_cfg" instead
  warnings.warn('DeprecationWarning: pretrained is a deprecated, '
/data02/xujunkai/EDEN/tasks/Segmentation/mmsegmentation/mmseg/models/losses/cross_entropy_loss.py:250: UserWarning: Default ``avg_non_ignore`` is False, if you would like to ignore the certain label and average loss over non-ignore labels, which is the same with PyTorch official cross_entropy, set ``avg_non_ignore=True``.
  warnings.warn(


Loads checkpoint by local backend from path: work_dirs/eden/fcn_diffae_unet/iter_40000.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: diffae_ema_shadow, diffae_unet.time_embed.0.weight, diffae_unet.time_embed.0.bias, diffae_unet.time_embed.2.weight, diffae_unet.time_embed.2.bias, diffae_unet.input_conv.weight, diffae_unet.input_conv.bias, diffae_unet.down_blocks.0.norm1.weight, diffae_unet.down_blocks.0.norm1.bias, diffae_unet.down_blocks.0.conv1.weight, diffae_unet.down_blocks.0.conv1.bias, diffae_unet.down_blocks.0.time_emb_proj.1.weight, diffae_unet.down_blocks.0.time_emb_proj.1.bias, diffae_unet.down_blocks.0.cond_emb_proj.1.weight, diffae_unet.down_blocks.0.cond_emb_proj.1.bias, diffae_unet.down_blocks.0.norm2.weight, diffae_unet.down_blocks.0.norm2.bias, diffae_unet.down_blocks.0.conv2.weight, diffae_unet.down_blocks.0.conv2.bias, diffae_unet.down_blocks.1.op.weight, diffae_unet.down_blocks.1.op.bias, diffae_unet.down_blocks.2.norm1

/data02/xujunkai/EDEN/visual/probe_core.py:371: UserWarning: The label '_child1' of <matplotlib.lines.Line2D object at 0x7fdb67bb2190> starts with '_'. It is thus excluded from the legend.
  ax.legend(lines, [l.get_label() for l in lines], fontsize=8)
/data02/xujunkai/EDEN/visual/probe_core.py:371: UserWarning: The label '_child1' of <matplotlib.lines.Line2D object at 0x7fdb67b9f8e0> starts with '_'. It is thus excluded from the legend.
  ax.legend(lines, [l.get_label() for l in lines], fontsize=8)


iter_40000: Spearman ρ=nan (p=nan), Pearson r=nan
Loads checkpoint by local backend from path: work_dirs/eden/fcn_diffae_unet/iter_56000.pth
The model and loaded state dict do not match exactly

unexpected key in source state_dict: diffae_ema_shadow, diffae_unet.time_embed.0.weight, diffae_unet.time_embed.0.bias, diffae_unet.time_embed.2.weight, diffae_unet.time_embed.2.bias, diffae_unet.input_conv.weight, diffae_unet.input_conv.bias, diffae_unet.down_blocks.0.norm1.weight, diffae_unet.down_blocks.0.norm1.bias, diffae_unet.down_blocks.0.conv1.weight, diffae_unet.down_blocks.0.conv1.bias, diffae_unet.down_blocks.0.time_emb_proj.1.weight, diffae_unet.down_blocks.0.time_emb_proj.1.bias, diffae_unet.down_blocks.0.cond_emb_proj.1.weight, diffae_unet.down_blocks.0.cond_emb_proj.1.bias, diffae_unet.down_blocks.0.norm2.weight, diffae_unet.down_blocks.0.norm2.bias, diffae_unet.down_blocks.0.conv2.weight, diffae_unet.down_blocks.0.conv2.bias, diffae_unet.down_blocks.1.op.weight, diffae_unet.down

KeyboardInterrupt: 

In [ ]:
# 跨 ckpt 汇总：相关性随训练进度演化
import matplotlib.pyplot as plt
if len(rows) > 1:
    tags = [r[0] for r in rows]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(tags, [r[1] for r in rows], 'o-', label='Spearman ρ')
    ax.plot(tags, [r[3] for r in rows], 's-', label='Pearson r')
    ax.axhline(0, color='k', ls='--', lw=0.8)
    ax.set_ylim(-1, 1); ax.grid(alpha=0.3); ax.legend()
    ax.set_title('Dt ~ meta-grad correlation across checkpoints')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.savefig(os.path.join(OUT, 'summary.png'), dpi=150, bbox_inches='tight')
    fig